# Consensus Principle Selection

In [1]:
from pathlib import Path
from typing import Dict, List, Tuple

import time
import sys
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
DATA_PATH = Path(r"C:\Users\laras\OneDrive\Dokumente\duke_classes\icai_project\Principle-Elicitation\DecontextualPrinciples\self_written_ai_principles_th=0.42_summarized_complete_linkage.json") 

TOP_K_SUMMARIES = 5

TRESHOLD = "0.42"

# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)

In [3]:
def embed_texts(texts):
    single_input = isinstance(texts, str)
    if single_input:
        texts = [texts]

    batch_size = 256  
    all_vectors = []
    max_retries = 5

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        num_retries = 0
        while True:
            try:
                resp = client.embeddings.create(
                    model="text-embedding-3-small",
                    input=batch,
                )
                all_vectors.extend([item.embedding for item in resp.data])
                break
            except Exception as e:
                num_retries += 1
                print(f"Error: {e}. Retrying in 5 seconds...")
                if num_retries >= max_retries:
                    print("Max retries reached. Exiting.")
                    sys.exit(1)
                time.sleep(5)

    arr = np.array(all_vectors, dtype=float)

    return arr

In [4]:
# Load clustered principles
with DATA_PATH.open("r", encoding="utf-8") as f:
    principles_data = json.load(f)

summaries = []
summary_cluster_ids = []
originals = []

for idx, item in enumerate(principles_data):
    cluster_id = item.get("cluster_id")

    summarized_principle = item.get("summarized_principle").strip()
    principle_list = item.get("original_principles", [])

    if summarized_principle:
        summaries.append(summarized_principle)
        summary_cluster_ids.append(cluster_id)

    for p in principle_list:
        p = (p or "").strip()
        if p:
            originals.append(p)

print(f"#summaries: {len(summaries)}")
print(f"#originals: {len(originals)}")


#summaries: 276
#originals: 4115


In [5]:
emb_orig = embed_texts(originals)  
emb_summaries = embed_texts(summaries)  

N, d = emb_orig.shape if emb_orig.size else (0, 0)
M, d_s = emb_summaries.shape if emb_summaries.size else (0, 0)

print(f"emb_orig shape: {emb_orig.shape}")
print(f"emb_summaries shape: {emb_summaries.shape}")

emb_orig shape: (4115, 1536)
emb_summaries shape: (276, 1536)


In [6]:
global_msd = []  

for j in range(M):
    s_j = emb_summaries[j] 
    diff = emb_orig - s_j   
    msd_j = (diff ** 2).mean() 
    global_msd.append(float(msd_j))

global_msd = np.array(global_msd)
global_msd[:10]  

array([0.00052676, 0.00051484, 0.00051369, 0.00063363, 0.00046675,
       0.00047461, 0.00054521, 0.00051511, 0.00058207, 0.00053905])

In [7]:
idx_sorted = np.argsort(global_msd)

rows = []
for rank, j in enumerate(idx_sorted, start=1):
    rows.append(
        {
            "rank": rank,
            "cluster_id": summary_cluster_ids[j],
            "summary_text": summaries[j],
            "global_msd": float(global_msd[j]),
        }
    )

df_summaries = pd.DataFrame(rows)
df_summaries.head(TOP_K_SUMMARIES)

,rank,cluster_id,summary_text,global_msd
0,1,62,AI should be honest and truthful.,0.000429
1,2,81,AI should provide truthful and accurate inform...,0.000434
2,3,107,AI should avoid bias.,0.000435
3,4,124,AI should be kind and non-judgemental in its i...,0.000443
4,5,113,AI should provide factual and accurate informa...,0.000449


In [8]:
output_json_path = Path(f"consensus_principle_selection_th={TRESHOLD}_complete_linkage.json")
with output_json_path.open("w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

print("Saved JSON to:", output_json_path)

Saved JSON to: consensus_principle_selection_th=0.42_complete_linkage.json


In [9]:
top = df_summaries.head(TOP_K_SUMMARIES)
for _, row in top.iterrows():
    print("Rank:", row["rank"])
    print("Cluster:", row["cluster_id"])
    print("Global MSD:", row["global_msd"])
    print("Summary:\n", row["summary_text"])
    print("-" * 80)

Rank: 1
Cluster: 62
Global MSD: 0.0004292603555339397
Summary:
 AI should be honest and truthful.
--------------------------------------------------------------------------------
Rank: 2
Cluster: 81
Global MSD: 0.00043424989305303115
Summary:
 AI should provide truthful and accurate information.
--------------------------------------------------------------------------------
Rank: 3
Cluster: 107
Global MSD: 0.00043528521633069537
Summary:
 AI should avoid bias.
--------------------------------------------------------------------------------
Rank: 4
Cluster: 124
Global MSD: 0.0004429648019109595
Summary:
 AI should be kind and non-judgemental in its interactions.
--------------------------------------------------------------------------------
Rank: 5
Cluster: 113
Global MSD: 0.0004485807472906802
Summary:
 AI should provide factual and accurate information.
--------------------------------------------------------------------------------
